In [ ]:
!pip install -U transformers
!pip install evaluate accelerate

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer

from datasets import load_dataset
import torch


In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
checkpoint = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2, id2label=id2label, label2id=label2id).to(device)

In [ ]:
raw_dataset = load_dataset("imdb")
raw_dataset.shape

In [ ]:
def tokenizer_function(example):
  return tokenizer(example["text"], truncation = True)
tokenizer_dataset = raw_dataset.map(tokenizer_function, batched = True)
data_collate =DataCollatorWithPadding(tokenizer = tokenizer)


In [ ]:
import evaluate
import numpy as np
accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pre):
  predictions, labels = eval_pre
  predictions = np.argmax(predictions, axis= 1)
  return accuracy.compute(predictions = predictions, references= labels)

In [ ]:
training_args = TrainingArguments(
    output_dir="bert_fine_tune",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=True,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenizer_dataset["train"],
    eval_dataset=tokenizer_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collate,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
trainer.push_to_hub()